In [1]:
!pip install -q -r requirements.txt

In [2]:
import os
import json
import time
from dotenv import load_dotenv
import chromadb
from google import genai
from google.genai import errors

load_dotenv()

api_key = os.environ.get("GOOGLE_API_KEY")
assert api_key, "GOOGLE_API_KEY not set — check your .env file"
print(f"Key loaded: {api_key[:8]}...{api_key[-4:]}")

client = genai.Client(api_key=api_key)
print("Gemini client ready.")


def call_with_retry(fn, max_attempts=3, delay=5):
    """Retry on transient server errors (503). Re-raises on real failures."""
    for attempt in range(max_attempts):
        try:
            return fn()
        except errors.ServerError as e:
            print(f"  Server busy (attempt {attempt + 1}/{max_attempts}): {e}")
            if attempt < max_attempts - 1:
                time.sleep(delay)
            else:
                raise

Key loaded: AQ.Ab8RN...BMTg
Gemini client ready.


In [3]:
with open("knowledge_base.json", "r") as f:
    entries = json.load(f)

print(f"Loaded {len(entries)} entries\n")
for e in entries:
    print(f"  [{e['id']}]  {e['source']:<16}  {e['text'][:72]}...")

Loaded 10 entries

  [kb-01]  handbook.md       Employees may park in lot B after 6pm on weekdays. Lot A is reserved for...
  [kb-02]  handbook.md       To power up a device that won't turn on, hold the power button for ten s...
  [kb-03]  handbook.md       Annual leave must be requested at least two weeks in advance through the...
  [kb-04]  policy.md         Our refund policy allows a full refund within 30 days of purchase, provi...
  [kb-05]  policy.md         To cancel your subscription, open Account Settings and choose End Plan. ...
  [kb-06]  policy.md         Premium plan members get priority support, with a guaranteed first respo...
  [kb-07]  it.md             Reset your password from the login screen by clicking 'Forgot password'....
  [kb-08]  it.md             The error code 0x80070005 means 'access denied'. Run the application as ...
  [kb-09]  it.md             Company laptops back up automatically to the cloud every night at 2am wh...
  [kb-10]  facilities.md     The off

In [7]:
# ── Stage: Index ──────────────────────────────────────────────────────

chroma_client = chromadb.EphemeralClient()


def get_embedding(text):
    def _call():
        result = client.models.embed_content(
            model="gemini-embedding-001",
            contents=text
        )
        return result.embeddings[0].values

    return call_with_retry(_call)


def index_knowledge_base(entries):
    collection = chroma_client.get_or_create_collection(
        name="kb_collection",
        metadata={"hnsw:space": "cosine"}
  )

    ids        = [e["id"]     for e in entries]
    documents  = [e["text"]   for e in entries]
    metadatas  = [{"source": e["source"]} for e in entries]
    embeddings = [get_embedding(e["text"]) for e in entries]

    collection.upsert(
        ids=ids,
        documents=documents,
        embeddings=embeddings,
        metadatas=metadatas,
    )

    return collection


collection = index_knowledge_base(entries)
print(f"Collection ready — {collection.count()} passages indexed.")

Collection ready — 10 passages indexed.


In [9]:
# ── Stage: Query ──────────────────────────────────────────────────────

def query(question, top_k=3):
    q_embedding = get_embedding(question)

    results = collection.query(
        query_embeddings=[q_embedding],
        n_results=top_k,
        include=["documents", "metadatas", "distances"]
    )

    passages = []
    for doc, meta in zip(results["documents"][0], results["metadatas"][0]):
        passages.append({"text": doc, "source": meta["source"]})

    return passages

In [11]:
# ── Stage: Generate ───────────────────────────────────────────────────

def assemble_prompt(question, passages):
    context_block = ""
    for p in passages:
        context_block += f"[Source: {p['source']}]\n{p['text']}\n\n"

    prompt = (
        "Answer the question using only the context below.\n"
        "If the answer is not present in the context, say exactly "
        "\"I don't know\" — do not speculate and do not use outside knowledge.\n"
        "For every fact in your answer, cite the source in brackets, "
        "e.g. [Source: policy.md].\n\n"
        f"Context:\n{context_block.strip()}\n\n"
        f"Question: {question}\n"
        "Answer:"
    )

    return prompt


def generate(question, top_k=3):
    passages = query(question, top_k=top_k)
    prompt   = assemble_prompt(question, passages)

    def _call():
        return client.models.generate_content(
            model="gemini-2.5-flash",
            contents=prompt
        )

    response = call_with_retry(_call)

    return {
        "question": question,
        "passages": passages,
        "answer":   response.text.strip()
    }

In [12]:
questions = [
    "How long do I have to get a full refund?",
    "How do I reset my password?",
    "What is the company's stock price today?",
]

results = []
for q in questions:
    print(f"Running: {q}")
    result = generate(q)
    results.append(result)
    print("  done.\n")

Running: How long do I have to get a full refund?
  done.

Running: How do I reset my password?
  done.

Running: What is the company's stock price today?
  done.



In [13]:
for r in results:
    print(f"Q: {r['question']}")
    print("-" * 60)
    print("Retrieved passages:")
    for p in r['passages']:
        print(f"  [{p['source']}] {p['text']}")
    print(f"\nAnswer: {r['answer']}")
    print("=" * 60)
    print()

Q: How long do I have to get a full refund?
------------------------------------------------------------
Retrieved passages:
  [policy.md] Our refund policy allows a full refund within 30 days of purchase, provided the item is unused and in its original packaging. After 30 days, only store credit is offered.
  [policy.md] To cancel your subscription, open Account Settings and choose End Plan. Cancellation takes effect at the end of the current billing period; no partial refunds are given for the remaining days.
  [policy.md] Premium plan members get priority support, with a guaranteed first response within four business hours. Standard members receive a response within two business days.

Answer: You have 30 days of purchase to get a full refund [Source: policy.md].

Q: How do I reset my password?
------------------------------------------------------------
Retrieved passages:
  [it.md] Reset your password from the login screen by clicking 'Forgot password'. A reset link is emailed to 

In [14]:
stretch_result = generate("How long do I have to get a full refund?", top_k=1)

print(f"Q: {stretch_result['question']} (top_k=1)")
print("-" * 60)
print("Retrieved passages:")
for p in stretch_result['passages']:
    print(f"  [{p['source']}] {p['text']}")
print(f"\nAnswer: {stretch_result['answer']}")

Q: How long do I have to get a full refund? (top_k=1)
------------------------------------------------------------
Retrieved passages:
  [policy.md] Our refund policy allows a full refund within 30 days of purchase, provided the item is unused and in its original packaging. After 30 days, only store credit is offered.

Answer: You have 30 days of purchase to get a full refund [Source: policy.md].


> With `top_k=1`, the answer was identical because the refund passage was already the single highest-ranked match; the risk only appears when the most relevant passage isn't ranked first — too little context then drops something the answer actually needs, while too much context (`top_k=10` or more) risks the model blending unrelated passages into a less grounded answer.

> The refund answer cited `policy.md` directly, pulling the exact 30-day window and store-credit fallback from the top-ranked passage. The password reset answer cited `it.md`, restating the "Forgot password" flow and one-hour link expiry faithfully. The stock price question was declined with a literal "I don't know" — no hedging, no fallback to general knowledge, even though three policy/handbook passages were retrieved (none relevant, since nothing in the KB covers stock price). Dropping to `top_k=1` on the refund question produced an identical answer, because the correct passage was already ranked first at `top_k=3` — the retrieval didn't need the extra two passages to land on the right answer this time.

> Yesterday's NumPy + cosine similarity approach required holding every embedding in memory and computing similarity against the full corpus by hand on each query — fine for 10 passages, but it doesn't scale to a real knowledge base with thousands of documents the way Chroma's indexed vector store does. On top of that scalable retrieval, the citation + refusal pattern is what turns raw passage-matching into something trustworthy: tagging sources at index time and forcing an explicit "I don't know" instruction means the system can be checked and trusted rather than just returning text and hoping it's grounded.